# SNAP — Training Pipeline
Carica i CSV già pronti, costruisce la feature matrix e allena XGBoost con walk-forward validation.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import xgboost as xgb
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

DATA_DIR   = './data'
MODELS_DIR = './models'
os.makedirs(MODELS_DIR, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e',
    'text.color': '#c9d1d9', 'grid.color': '#21262d',
    'grid.linestyle': '--', 'grid.alpha': 0.6,
    'legend.facecolor': '#161b22', 'legend.edgecolor': '#30363d',
})
print('✅ Librerie caricate.')

## 1. Caricamento CSV

In [ ]:
# Mercato
df_market = pd.read_csv(
    os.path.join(DATA_DIR, 'btc-usd_market_data.csv'),
    index_col='timestamp', parse_dates=True
)
print(f'📈 Mercato:  {len(df_market)} settimane | {df_market.index[0].date()} → {df_market.index[-1].date()}')

# Mining
df_mining = pd.read_csv(
    os.path.join(DATA_DIR, 'bitcoin_mining_metrics.csv'),
    index_col='timestamp', parse_dates=True
)
print(f'⛏️  Mining:   {len(df_mining)} settimane')

# Energia
df_energy = pd.read_csv(
    os.path.join(DATA_DIR, 'bitcoin_energy_cost.csv'),
    index_col='timestamp', parse_dates=True
)
print(f'⚡ Energia:  {len(df_energy)} settimane | colonne: {list(df_energy.columns)}')

# Sentiment (già calcolato da Gemini)
df_sentiment = pd.read_csv(
    os.path.join(DATA_DIR, 'bitcoin_sentiment.csv'),
    index_col='timestamp', parse_dates=True
)
print(f'🐦 Sentiment: {len(df_sentiment)} settimane | dati reali: {df_sentiment["tweet_count"].gt(0).sum()}')

# Fear & Greed Index
import requests as _req
try:
    resp = _req.get("https://api.alternative.me/fng/?limit=1000&format=json", timeout=30)
    resp.raise_for_status()
    fng_data = resp.json()["data"]
    df_fng = pd.DataFrame(fng_data)
    df_fng["timestamp"] = pd.to_datetime(df_fng["timestamp"].astype(int), unit="s")
    df_fng = df_fng.set_index("timestamp")[["value"]].sort_index()
    df_fng["value"] = pd.to_numeric(df_fng["value"])
    df_fng_weekly = df_fng["value"].resample("W").agg(
        fear_greed_mean  = "mean",
        fear_greed_close = "last",
    )
    df_fng_weekly["fear_greed_delta"] = df_fng_weekly["fear_greed_close"].diff()
    fng_path = os.path.join(DATA_DIR, "bitcoin_fear_greed.csv")
    df_fng_weekly.to_csv(fng_path)
    print(f'😱 Fear & Greed: {len(df_fng_weekly)} settimane salvate in {fng_path}')
except Exception as e:
    print(f"[warn] Fear & Greed non disponibile: {e}")
    # Carica da CSV se già scaricato
    fng_path = os.path.join(DATA_DIR, "bitcoin_fear_greed.csv")
    if os.path.exists(fng_path):
        df_fng_weekly = pd.read_csv(fng_path, index_col="timestamp", parse_dates=True)
        print(f"😱 Fear & Greed caricato da CSV: {len(df_fng_weekly)} settimane")
    else:
        df_fng_weekly = None
        print("[warn] Fear & Greed non disponibile — verrà escluso dalla feature matrix")

df_sentiment.head()

## 2. Feature Engineering

In [ ]:
from src.sentiment_engine import SentimentEngine
from dotenv import load_dotenv
load_dotenv()

engine = SentimentEngine(api_key=os.getenv('GEMINI_API_KEY'))
print('✅ SentimentEngine pronto.')

In [ ]:
# Feature tecniche
tech = engine.compute_technical_features(df_market)
print(f'🔧 Tech features: {tech.shape[1]} colonne, {tech.shape[0]} settimane')

In [ ]:
# Feature energia
energy_feats = engine.compute_energy_features(df_energy)
print(f'⚡ Energy features: {energy_feats.shape[1]} colonne, {energy_feats.shape[0]} settimane')

In [ ]:
# Gap filling sentiment (forward fill only — no look-ahead)
full_idx = pd.date_range(
    start=df_sentiment.index.min(),
    end=df_sentiment.index.max(),
    freq='W',
)
df_sentiment = df_sentiment.reindex(full_idx)
df_sentiment['has_data']    = (df_sentiment['tweet_count'] > 0).astype(int)
df_sentiment['tweet_count'] = df_sentiment['tweet_count'].fillna(0)

cols_interp = ['sentiment_score_mean', 'sentiment_score_weighted', 'positive_pct', 'negative_pct']
df_sentiment[cols_interp] = df_sentiment[cols_interp].ffill(limit=4).fillna(0)

print(f'Settimane con dati reali:  {df_sentiment["has_data"].sum()}')
print(f'Settimane interpolate:     {(df_sentiment["has_data"] == 0).sum()}')

In [ ]:
# Feature matrix
feature_matrix = tech.join(df_sentiment, how='left').join(energy_feats, how='left')

# Aggiungi Fear & Greed se disponibile
if df_fng_weekly is not None:
    feature_matrix = feature_matrix.join(df_fng_weekly, how='left')
    feature_matrix[['fear_greed_mean', 'fear_greed_close', 'fear_greed_delta']] = (
        feature_matrix[['fear_greed_mean', 'fear_greed_close', 'fear_greed_delta']]
        .ffill(limit=1)
    )
    print("😱 Fear & Greed aggiunto alla feature matrix")

feature_matrix = feature_matrix.ffill(limit=1)
feature_matrix = feature_matrix.iloc[:-1]  # ultima settimana non ancora chiusa

print(f'📊 Feature matrix: {feature_matrix.shape[0]} settimane × {feature_matrix.shape[1]} feature')
print(f'   Periodo: {feature_matrix.index[0].date()} → {feature_matrix.index[-1].date()}')
print()
for col in feature_matrix.columns:
    na = feature_matrix[col].isna().sum()
    print(f'  {col:<35} NaN: {na}')

## 3. Target

In [ ]:
close = df_market['Close']
target = []
for t in feature_matrix.index:
    t_idx = close.index.get_loc(t)
    up = int(close.iloc[t_idx + 1] > close.iloc[t_idx]) if t_idx + 1 < len(close) else np.nan
    target.append(up)

y = pd.Series(target, index=feature_matrix.index, name='target_up')
print('Target distribution:')
print(y.value_counts())
print(f'\nUP ratio: {y.mean():.1%}')

## 4. Preparazione X/y

In [ ]:
X = feature_matrix.dropna()
y_clean = y.loc[X.index].dropna()
X_clean = X.loc[y_clean.index]

split   = int(len(X_clean) * 0.7)
X_train = X_clean.iloc[:split]
y_train = y_clean.iloc[:split]
X_test  = X_clean.iloc[split:]
y_test  = y_clean.iloc[split:]

print(f'Dataset: {len(X_clean)} settimane totali')
print(f'Train:   {len(X_train)} settimane | Test: {len(X_test)} settimane')

## 5. Hyperparameter Tuning

In [ ]:
param_grid = {
    'n_estimators':     [50, 100, 200, 300],
    'max_depth':        [2, 3, 4, 5],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'subsample':        [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma':            [0, 0.1, 0.3],
    'reg_alpha':        [0, 0.1, 0.5],
    'reg_lambda':       [1, 2, 5],
}

tscv = TimeSeriesSplit(n_splits=5)
search = RandomizedSearchCV(
    estimator=xgb.XGBClassifier(eval_metric='logloss', random_state=42),
    param_distributions=param_grid,
    n_iter=50,
    scoring='roc_auc',
    cv=tscv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

best_params = search.best_params_
print(f'Migliori parametri: {best_params}')
print(f'Miglior AUC in CV:  {search.best_score_:.3f}')

## 6. Walk-Forward Validation

In [ ]:
MIN_TRAIN_SIZE = 30

predictions   = []
probabilities = []
actuals       = []
test_dates    = []

for i in range(MIN_TRAIN_SIZE, len(X_clean)):
    X_tr = X_clean.iloc[:i]
    y_tr = y_clean.iloc[:i]
    X_te = X_clean.iloc[i:i+1]
    y_te = y_clean.iloc[i:i+1]

    model = xgb.XGBClassifier(**best_params, eval_metric='logloss', random_state=42)
    model.fit(X_tr, y_tr)

    predictions.append(model.predict(X_te)[0])
    probabilities.append(model.predict_proba(X_te)[0, 1])
    actuals.append(y_te.values[0])
    test_dates.append(X_te.index[0])

predictions   = np.array(predictions)
probabilities = np.array(probabilities)
actuals       = np.array(actuals)

acc = accuracy_score(actuals, predictions)
auc = roc_auc_score(actuals, probabilities)

print(f'📊 Walk-Forward Validation (expanding window):')
print(f'   Predizioni totali: {len(predictions)}')
print(f'   Accuracy : {acc:.3f}')
print(f'   ROC-AUC  : {auc:.3f}')

## 7. Baseline di confronto

In [ ]:
# Buy & Hold
buy_hold_acc = accuracy_score(actuals, np.ones(len(actuals), dtype=int))

# MACD Crossover
macd_preds = (X_clean['macd'] > X_clean['macd_signal']).astype(int).values[MIN_TRAIN_SIZE:]
macd_diff  = (X_clean['macd'] - X_clean['macd_signal']).values[MIN_TRAIN_SIZE:]
macd_prob  = (macd_diff - macd_diff.min()) / (macd_diff.max() - macd_diff.min() + 1e-9)
macd_acc   = accuracy_score(actuals, macd_preds)
macd_auc   = roc_auc_score(actuals, macd_prob)

# Random (media 50 seed)
random_aucs, random_accs = [], []
for seed in range(50):
    rng = np.random.RandomState(seed)
    random_aucs.append(roc_auc_score(actuals, rng.uniform(0, 1, len(actuals))))
    random_accs.append(accuracy_score(actuals, rng.randint(0, 2, len(actuals))))
random_aucs = np.array(random_aucs)
random_accs = np.array(random_accs)

print(f'📊 Confronto Walk-Forward\n')
print(f'{"Strategia":<20} {"Accuracy":<12} {"ROC-AUC"}')
print('-' * 42)
print(f'{"XGBoost":<20} {acc:<12.3f} {auc:.3f}')
print(f'{"Buy & Hold":<20} {buy_hold_acc:<12.3f} {"N/A"}')
print(f'{"MACD Crossover":<20} {macd_acc:<12.3f} {macd_auc:.3f}')
print(f'{"Random (media)":<20} {random_accs.mean():<12.3f} {random_aucs.mean():.3f}')

## 8. Salvataggio modello

In [ ]:
# Allena il modello finale su tutti i dati disponibili
final_model = xgb.XGBClassifier(**best_params, eval_metric='logloss', random_state=42)
final_model.fit(X_clean, y_clean)

model_path = os.path.join(MODELS_DIR, 'xgboost_model.json')
final_model.save_model(model_path)

# Salva anche le feature names per il predictor
import json
with open(os.path.join(MODELS_DIR, 'feature_names.json'), 'w') as f:
    json.dump(X_clean.columns.tolist(), f)

print(f'✅ Modello salvato in {model_path}')
print(f'   Feature: {X_clean.shape[1]}')
print(f'   Trained on: {len(X_clean)} settimane')

## 9. Backtesting

In [ ]:
close = df_market['Close']

bt = pd.DataFrame({
    'prediction':  predictions,
    'probability': probabilities,
    'actual':      actuals,
}, index=test_dates)

bt['close_t']  = close.reindex(bt.index).values
bt['close_t1'] = [close.iloc[close.index.get_loc(t) + 1] for t in bt.index]
bt['return_w'] = bt['close_t1'] / bt['close_t'] - 1

CAPITAL_START   = 10_000.0
TRANSACTION_FEE = 0.001

capital_model = CAPITAL_START
capital_bh    = CAPITAL_START
equity_model  = []
equity_bh     = []
trades        = 0
in_position   = False

for _, row in bt.iterrows():
    ret = row['return_w']
    capital_bh *= (1 + ret)
    if row['prediction'] == 1:
        if not in_position:
            capital_model *= (1 - TRANSACTION_FEE)
            in_position = True
            trades += 1
        capital_model *= (1 + ret)
    else:
        if in_position:
            capital_model *= (1 - TRANSACTION_FEE)
            in_position = False
    equity_model.append(capital_model)
    equity_bh.append(capital_bh)

bt['equity_model'] = equity_model
bt['equity_bh']    = equity_bh

ret_model = bt['equity_model'].iloc[-1] / CAPITAL_START - 1
ret_bh    = bt['equity_bh'].iloc[-1]    / CAPITAL_START - 1

def sharpe(eq): 
    r = eq.pct_change().dropna()
    return r.mean() / r.std() * np.sqrt(52)

def max_drawdown(eq):
    return ((eq - eq.cummax()) / eq.cummax()).min()

print(f'📊 Backtesting — Risultati\n')
print(f'  Numero di trade: {trades}')
print(f'{"Metrica":<28} {"XGBoost":<15} {"Buy & Hold"}')
print('-' * 55)
print(f'{"Rendimento totale":<28} {ret_model*100:>+.1f}%          {ret_bh*100:>+.1f}%')
print(f'{"Capitale finale":<28} ${capital_model:>10,.0f}    ${capital_bh:>10,.0f}')
print(f'{"Sharpe Ratio (ann.)":<28} {sharpe(bt["equity_model"]):>10.3f}    {sharpe(bt["equity_bh"]):>10.3f}')
print(f'{"Max Drawdown":<28} {max_drawdown(bt["equity_model"])*100:>+.1f}%          {max_drawdown(bt["equity_bh"])*100:>+.1f}%')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                          gridspec_kw={'height_ratios': [3, 1]})
fig.suptitle('Backtesting — XGBoost vs Buy & Hold', fontsize=14,
             fontweight='bold', color='#f0f6fc')

axes[0].plot(bt.index, bt['equity_model'], color='#79c0ff', linewidth=2,
             label=f'XGBoost ({ret_model*100:+.1f}%)')
axes[0].plot(bt.index, bt['equity_bh'],    color='#ffa657', linewidth=2,
             label=f'Buy & Hold ({ret_bh*100:+.1f}%)')
axes[0].axhline(CAPITAL_START, color='#8b949e', linewidth=0.8, linestyle=':')
axes[0].set_ylabel('Capitale (USD)', fontsize=11)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
axes[0].legend(fontsize=10)
axes[0].grid(True)

colors_signal = ['#3fb950' if p == 1 else '#f85149' for p in bt['prediction']]
axes[1].bar(bt.index, bt['prediction'], color=colors_signal, width=5, alpha=0.8)
axes[1].set_ylabel('Segnale\n(1=Long, 0=Cash)', fontsize=9)
axes[1].set_ylim(-0.1, 1.3)
axes[1].set_yticks([0, 1])
axes[1].grid(True, axis='y')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'chart_backtesting.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grafico backtesting salvato.')